# FinBERT Sentiment Scoring 

**Academic research only — not investment advice.** This project compares model-derived news sentiment against market features for S&P 500 sector ETFs; nothing here is a trading signal.

Scores each distinct headline (prepared in `07a_llm_scoring_input_prep.ipynb`) once with **FinBERT** (`ProsusAI/finbert`), a transformer fine-tuned on financial text, to produce a sentiment score compared head-to-head against RavenPack's built-in `event_sentiment_score`.

**Why FinBERT (local, free).** Runs entirely on this machine, so the WRDS-licensed headlines never leave it (license-safe) and there is no API cost. It is the "specialised financial transformer" thread from the project's own literature review. This makes the comparison **FinBERT vs RavenPack** — two purpose-built financial sentiment scorers.

**Functional separation and leakage control.** FinBERT receives only the news text. It is not given prices, returns, tickers, publication dates, or future outcomes, and it is used only to classify the tone of the supplied text. Directional prediction is performed later by a separate walk-forward classifier. This design prevents direct target leakage during inference. However, because FinBERT is a pre-trained model, possible pretraining-related historical knowledge cannot be completely ruled out and will be treated as a project limitation.

**Score mapping.** FinBERT returns probabilities for positive / negative / neutral. We derive:
`sentiment = P(positive) − P(negative)` (a −1..+1 signal, directly comparable to RavenPack), `label = argmax`, `confidence = max probability`.

In [1]:
import time
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# --- paths (work whether run from repo root or data_collection/) --------------
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "data_collection").exists() else CWD.parent
RAW_DIR = REPO_ROOT / "data_collection" / "raw"

SCORING_INPUT_CSV = RAW_DIR / "llm_scoring_input.csv"   # from 07a (distinct texts)
SCORES_CSV = RAW_DIR / "finbert_scores.csv"             # this notebook writes here

MODEL_NAME = "ProsusAI/finbert"
INFER_BATCH = 64        # texts per forward pass
SAVE_EVERY = 20_000     # append to disk this often (resume safety)
MAX_LEN = 128           # headlines are short; 128 tokens is ample

# GPU if a CUDA build of torch is installed. The default pip torch is CPU-only;
# to use an NVIDIA GPU, reinstall a CUDA build, e.g.:
#   pip install --index-url https://download.pytorch.org/whl/cu128 torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cpu":
    print("  (running on CPU — works, just slower. See the pip command above for GPU.)")
print(f"scoring input: {SCORING_INPUT_CSV} (exists: {SCORING_INPUT_CSV.exists()})")

torch 2.10.0+cpu | device: cpu
  (running on CPU — works, just slower. See the pip command above for GPU.)
scoring input: c:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\llm_scoring_input.csv (exists: True)


## 1. Load FinBERT

Downloads `ProsusAI/finbert` from Hugging Face on first run (~440 MB, public — no token needed) and caches it. We read the label order from the model config rather than hard-coding indices, so the positive/negative/neutral mapping is always correct.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()

# id2label from the model config, lowercased -> {0:'positive',1:'negative',2:'neutral'}
ID2LABEL = {i: lbl.lower() for i, lbl in model.config.id2label.items()}
LABELS = [ID2LABEL[i] for i in range(len(ID2LABEL))]
print("label order:", LABELS)
assert set(LABELS) == {"positive", "negative", "neutral"}, LABELS

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

label order: ['positive', 'negative', 'neutral']


## 2. Scoring function

Tokenize a batch of headlines, run one forward pass, softmax to probabilities, then map to `label` / `sentiment` / `confidence`. `torch.no_grad()` keeps memory low and inference fast.

In [3]:
@torch.no_grad()
def score_batch(texts: list) -> list:
    """Return [{label, sentiment, confidence}, ...] for a list of headline strings."""
    enc = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN
    ).to(DEVICE)
    probs = F.softmax(model(**enc).logits, dim=-1).cpu()  # (n, 3)

    pos_i = LABELS.index("positive")
    neg_i = LABELS.index("negative")
    out = []
    for row in probs:
        p = {LABELS[i]: float(row[i]) for i in range(len(LABELS))}
        top = max(p, key=p.get)
        out.append({
            "label": top,
            "sentiment": p["positive"] - p["negative"],  # -1..+1, comparable to RavenPack
            "confidence": row.max().item(),
        })
    return out

## 3. Sample run — sanity-check the scores

Score the first few headlines and eyeball that the labels and signs look right before committing to the full run.

In [4]:
SAMPLE_SIZE = 12
scoring_input = pd.read_csv(SCORING_INPUT_CSV)

sample = scoring_input.head(SAMPLE_SIZE)
sample_scores = pd.DataFrame(score_batch(sample["headline"].tolist()))
sample_scores.insert(0, "headline", sample["headline"].values)
pd.set_option("display.max_colwidth", 70)
sample_scores

,headline,label,sentiment,confidence
0,Amundi US Inflation Expectations 10Y UCITS ETF Acc: Net Asset Valu...,neutral,-0.009383,0.935280
1,BOJ: Unsecured Overnight Call Rate Data,neutral,-0.019922,0.923992
2,BOJ: Unsecured Overnight Call Rate Data,neutral,-0.019922,0.923992
3,BOJ: Unsecured Overnight Call Rate Data,neutral,-0.019922,0.923992
4,BOJ: Unsecured Overnight Call Rate Data,neutral,-0.019922,0.923992
5,BOJ: Unsecured Overnight Call Rate Data,neutral,-0.019922,0.923992
6,Amundi US Inflation Expectations 10Y UCITS ETF GBP Hedged Dist: Ne...,neutral,-0.006816,0.933338
7,Press Release: Fitch Takes Various Rating Actions on U.S. Enhanced...,neutral,0.011446,0.929090
8,Fitch Takes Various Rating Actions on U.S. Enhanced Municipal Bond...,neutral,0.011318,0.944234
9,USDA U.S. Livestock Imports from Canada -2-,neutral,0.071321,0.887841


## 4. Full run — score every distinct headline (resumable)

Scores all texts not already in `finbert_scores.csv`, in GPU/CPU batches, appending to disk every `SAVE_EVERY` rows so an interruption loses at most that many. Re-running resumes where it left off. This is local and free — no rate limits, no cost.

In [ ]:
SCORE_COLS = ["text_id", "label", "sentiment", "confidence", "model", "scored_at"]


def done_text_ids() -> set:
    if SCORES_CSV.exists():
        return set(pd.read_csv(SCORES_CSV, usecols=["text_id"])["text_id"])
    return set()


def append_scores(rows: list) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows, columns=SCORE_COLS)
    df.to_csv(SCORES_CSV, mode="a", header=not SCORES_CSV.exists(), index=False)


todo = scoring_input[~scoring_input["text_id"].isin(done_text_ids())].reset_index(drop=True)
print(f"texts to score: {len(todo):,}")

buffer, t0 = [], time.time()
for start in range(0, len(todo), INFER_BATCH):
    chunk = todo.iloc[start:start + INFER_BATCH]
    scores = score_batch(chunk["headline"].tolist())
    now = pd.Timestamp.utcnow().isoformat()
    for tid, s in zip(chunk["text_id"], scores):
        buffer.append({"text_id": tid, **s, "model": MODEL_NAME, "scored_at": now})
    if len(buffer) >= SAVE_EVERY:
        append_scores(buffer)
        done = start + len(chunk)
        print(f"  {done:,}/{len(todo):,} scored ({done/(time.time()-t0):.0f}/s)")
        buffer = []
append_scores(buffer)
print(f"done in {time.time()-t0:.1f}s")

## 5. Validate 

Check coverage and the score distribution. 

In [5]:
scores = pd.read_csv(SCORES_CSV) if SCORES_CSV.exists() else pd.DataFrame(columns=SCORE_COLS)
n_input = len(scoring_input)
print(f"distinct texts:  {n_input:,}")
print(f"scored:          {len(scores):,}  ({len(scores)/n_input:.1%} coverage)")
print(f"duplicate text_ids in scores: {scores['text_id'].duplicated().sum()}")
print()
if len(scores):
    print("label distribution:")
    print(scores["label"].value_counts())
    print()
    print("sentiment summary (P(pos) - P(neg)):")
    print(scores["sentiment"].describe()[["mean", "std", "min", "max"]])
    display(scores.head())

distinct texts:  304,809
scored:          304,809  (100.0% coverage)
duplicate text_ids in scores: 0

label distribution:
label
positive    109534
neutral     103738
negative     91537
Name: count, dtype: int64

sentiment summary (P(pos) - P(neg)):
mean    0.055884
std     0.644303
min    -0.969652
max     0.944995
Name: sentiment, dtype: float64


,text_id,label,sentiment,confidence,model,scored_at
0,1e15f4ebe48fa51b,neutral,-0.009383,0.935280,ProsusAI/finbert,2026-07-23T18:52:26.238383+00:00
1,9999669a235a9606,neutral,-0.019922,0.923992,ProsusAI/finbert,2026-07-23T18:52:26.238383+00:00
2,a5151d436c47b000,neutral,-0.019922,0.923992,ProsusAI/finbert,2026-07-23T18:52:26.238383+00:00
3,79ac36b7aa6be844,neutral,-0.019922,0.923992,ProsusAI/finbert,2026-07-23T18:52:26.238383+00:00
4,4b73410a98f477df,neutral,-0.019922,0.923992,ProsusAI/finbert,2026-07-23T18:52:26.238383+00:00
